# Single-label SAE Feature Combination Analysis

This notebook implements SAE feature combination analysis within a single label (Safe/Unsafe):
1. Use Safe/Unsafe labels as binary classification labels
2. Calculate SAE feature distribution in PR space

## 1. Import and Configuration

In [ ]:
import os
import yaml
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from sae_tools.adapters.datasets import get_adapter
from sae_tools.analysis.artifacts import (
    build_generate_activations_command,
    find_latest_activation_file,
    require_activation_keys,
)
from sae_tools.model import (
    load_sae_predictions_pt,
    filter_data_by_label
)
from sae_tools.analysis.statistical import (
    build_sentence_feature_matrix_from_sparse,
    evaluate_features,
    print_metrics_overview,
    print_top_features,
    plot_pr_space
)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']


In [ ]:
BASE_DIR = Path.cwd()
MODEL_ROOT = os.getenv("MODEL_ROOT")
SAE_ROOT = os.getenv("SAE_ROOT")
DATASET_ROOT = os.getenv("DATASET_ROOT")

MODEL_PROFILE = os.getenv("MODEL_PROFILE", "qwen3-8b-guard")
SAE_PROFILE = os.getenv("SAE_PROFILE", "qwen-scope-qwen3-8b-l0-50")
LAYER = int(os.getenv("SAE_LAYER", "18"))
RESULTS_DIR = Path(os.getenv("RESULTS_DIR", str(BASE_DIR / "results")))

print(f"Results root: {RESULTS_DIR}")
print(f"Model profile: {MODEL_PROFILE}")
print(f"SAE profile: {SAE_PROFILE}")
print(f"Layer: {LAYER}")
print("=" * 80)


In [ ]:
# prompt
# DATASET_NAME = "ToxicChat"
# DATASET_NAME = "OpenAIMod"
DATASET_NAME = os.getenv("DATASET_NAME", "Aegis1.0")
# DATASET_NAME = "Aegis2.0"
# DATASET_NAME = "SimpleSafetyTest"
# DATASET_NAME = "HarmBench"
# DATASET_NAME = "WildGuardTest"

# response
# DATASET_NAME = "BeaverTails"
# DATASET_NAME = "BeaverTailsAugmented"
# DATASET_NAME = "Aegis2.0"
# DATASET_NAME = "SafeRLHF"
# DATASET_NAME = "WildGuardMix"

DATASETS_CONFIG = Path(os.getenv("DATASETS_CONFIG", str(BASE_DIR / "configs/datasets/datasets_prompt.yaml")))
with open(DATASETS_CONFIG, 'r', encoding='utf-8') as f:
    dataset_config = yaml.safe_load(f)
datasets = dataset_config.get('datasets', [])

DATASET_PATH = None
for dataset_info in datasets:
    dataset_name = dataset_info.get('name')
    if dataset_name == DATASET_NAME:
        DATASET_PATH = os.path.join(DATASET_ROOT, dataset_info.get('folder'))
        DATA_TYPE = dataset_info.get('type')
        LABEL_TYPE = f"{DATA_TYPE}_label"
        print(f"Dataset path: {DATASET_PATH}")
        print(f"Label type: {LABEL_TYPE}")
        break
if DATASET_PATH is None:
    raise ValueError(f"Dataset {DATASET_NAME} not found in {DATASETS_CONFIG}")

GENERATION_COMMAND = build_generate_activations_command(
    dataset_config=DATASETS_CONFIG,
    dataset_name=DATASET_NAME,
    output_dir=RESULTS_DIR,
    model_profile=MODEL_PROFILE,
    sae_profile=SAE_PROFILE,
    layer=LAYER,
)
PT_FILE = find_latest_activation_file(
    results_dir=RESULTS_DIR,
    dataset_name=DATASET_NAME,
    model_profile=MODEL_PROFILE,
    sae_profile=SAE_PROFILE,
    layer=LAYER,
    generate_command=GENERATION_COMMAND,
)
OUTPUT_DIR = PT_FILE.parents[1]
PT_DIR = PT_FILE.parent
CHARTS_DIR = OUTPUT_DIR / "charts"
SUMMARY_FILE = OUTPUT_DIR / "safe_feature_combination_summary.txt"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Activation file: {PT_FILE}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Charts directory: {CHARTS_DIR}")
print(f"Output file: {SUMMARY_FILE}")
print("=" * 80)

## 2. Load Data

Load SAE prediction results and metadata from PT files

In [ ]:
sparse_data = load_sae_predictions_pt(PT_FILE)
require_activation_keys(sparse_data)
num_samples = len(sparse_data['seq_lens'])

print(f"Loaded sparse activation data, number of samples: {num_samples}")
adapter = get_adapter(DATASET_NAME)
dataset = adapter.load(DATASET_PATH, num_samples)

metadata_list = [item for item in dataset]
print(f"Loaded {len(metadata_list)} metadata records")

In [ ]:
sparse_data

In [ ]:
if len(metadata_list) != num_samples:
    raise ValueError(f"Warning: Number of activation data samples ({num_samples}) does not match number of metadata samples ({len(metadata_list)})")
    
sparse_data, data_list, valid_indices = filter_data_by_label(
    sparse_data, 
    metadata_list, 
    label_field=LABEL_TYPE,
    verbose=True
)

print(f"Data list length after merging: {len(data_list)}")
print(f"\nFirst record's category field: {data_list[0].get('category', 'N/A')}")
print(f"First record's {LABEL_TYPE} field: {data_list[0].get(LABEL_TYPE, 'N/A')}")

## 3. Analyze PR Space Using Safe/Unsafe Labels

Directly use prompt_label/response_label (Safe/Unsafe) as binary classification labels to analyze the distribution of all SAE features in PR space

In [ ]:
# Build feature matrix (only take max from response start position)
X = build_sentence_feature_matrix_from_sparse(sparse_data)
y_unsafe = np.array([1 if item.get(LABEL_TYPE) == 'Unsafe' else 0 for item in data_list])

print(f"Feature matrix shape: {X.shape}")
print(f"Number of non-zero elements: {X.nnz}")
print(f"Sparsity: {1 - X.nnz / (X.shape[0] * X.shape[1]):.4%}")

In [ ]:
feature_activation_ratio = np.array(X.count_nonzero(axis=0)).flatten() / X.shape[0]
print(feature_activation_ratio.max())

In [ ]:
metrics_result = evaluate_features(
    X, y_unsafe,
    top_k=50,
    batch_size=1000
)
print_metrics_overview(metrics_result)
print_top_features(metrics_result, 'high_f1', 'Top 10 F1 Features')

In [ ]:
# Plot PR space chart (colored by feature_diff by default)
chart_pr_path = os.path.join(CHARTS_DIR, f"{DATASET_NAME}_pr_space.png")
plot_pr_space(metrics_result, output_file=chart_pr_path, color_by='diff', highlight_indices=[9249])# , 4744, 42948, 49513])
print(f"Chart saved to: {chart_pr_path}")

## 4. Summarize Top Features for Safe/Unsafe Labels

Display top F1/Precision/Recall features for Safe/Unsafe labels

In [ ]:
# Display comparison of Top Features across different metrics
print("=" * 80)
print("Top Features Comparison Across Different Metrics")
print("=" * 80)
# Print Top 10 Features for different metrics
print_top_features(metrics_result, 'high_f1', 'Top 10 F1 Features')

In [ ]:
print_top_features(metrics_result, 'high_precision', 'Top 10 Precision Features (recall > 0.1)')

In [ ]:
print_top_features(metrics_result, 'high_recall', 'Top 10 Recall Features')

In [ ]:
print_top_features(metrics_result, 'high_diff', 'Top 10 Diff Features')

In [ ]:
print_top_features(metrics_result, 'pareto', 'Top 10 Pareto Front Features')